In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mart_files = list(Path("data/mart").glob("*.csv"))
df = pd.read_csv(mart_files[0])
df['date'] = pd.to_datetime(df['date'])

print("=== Данные ===")
print(df.head())
print(df.columns.tolist())

features = ['avg_depth_km', 'earthquake_count', 'avg_magnitude', 'day_of_week', 'month']
target = 'max_magnitude'

df = df.dropna(subset=features + [target])
X = df[features].copy()
y = df[target].copy()

print(f"\nРазмер X: {X.shape}")
print(f"Размер y: {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"\nОбучающая выборка: {len(X_train)} строк")
print(f"Тестовая выборка: {len(X_test)} строк")

dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

print("\n=== BASELINE (предсказание средним) ===")
print(f"MAE: {mean_absolute_error(y_test, y_pred_dummy):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_dummy)):.3f}")
print(f"R2: {r2_score(y_test, y_pred_dummy):.3f}")

pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

pipeline_lr.fit(X_train, y_train)
y_pred_lr = pipeline_lr.predict(X_test)

print("\n=== ЛИНЕЙНАЯ РЕГРЕССИЯ ===")
print(f"MAE: {mean_absolute_error(y_test, y_pred_lr):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lr)):.3f}")
print(f"R2: {r2_score(y_test, y_pred_lr):.3f}")

pipeline_dt = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', DecisionTreeRegressor(max_depth=5, random_state=42))
])

pipeline_dt.fit(X_train, y_train)
y_pred_dt = pipeline_dt.predict(X_test)

print("\n=== ДЕРЕВО РЕШЕНИЙ ===")
print(f"MAE: {mean_absolute_error(y_test, y_pred_dt):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_dt)):.3f}")
print(f"R2: {r2_score(y_test, y_pred_dt):.3f}")

improvement_lr = (1 - mean_absolute_error(y_test, y_pred_lr) / mean_absolute_error(y_test, y_pred_dummy)) * 100
improvement_dt = (1 - mean_absolute_error(y_test, y_pred_dt) / mean_absolute_error(y_test, y_pred_dummy)) * 100

print("\n=== ВЫВОДЫ ===")
print(f"Baseline MAE: {mean_absolute_error(y_test, y_pred_dummy):.3f}")
print(f"Linear Regression MAE: {mean_absolute_error(y_test, y_pred_lr):.3f} (улучшение на {improvement_lr:.1f}%)")
print(f"Decision Tree MAE: {mean_absolute_error(y_test, y_pred_dt):.3f} (улучшение на {improvement_dt:.1f}%)")
print()
print("Модель незначительно лучше baseline. Магнитуда слабо зависит от выбранных признаков.")
print("Практическая ценность модели низкая.")

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred_lr, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel("Фактическая магнитуда")
plt.ylabel("Предсказанная магнитуда")
plt.title("Линейная регрессия")

plt.subplot(1, 2, 2)
plt.scatter(y_test, y_pred_dt, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel("Фактическая магнитуда")
plt.ylabel("Предсказанная магнитуда")
plt.title("Дерево решений")

plt.tight_layout()
plt.savefig("docs/ml/metrics.png", dpi=150)
plt.show()

results = pd.DataFrame({
    'actual': y_test.values,
    'predicted_linear': y_pred_lr,
    'predicted_tree': y_pred_dt,
    'error_linear': abs(y_test.values - y_pred_lr),
    'error_tree': abs(y_test.values - y_pred_dt)
})
results.head(10).to_csv("docs/ml/predictions_sample.csv", index=False)

print("\nФайлы сохранены:")
print("- docs/ml/metrics.png")
print("- docs/ml/predictions_sample.csv")